# Numerical propagation with propygator

This notebook walks through Feature 1.1: propagate an initial state vector with a
configurable force model, inspect the resulting `Trajectory`, and produce the
standard plot + CSV outputs.

Importing `propygator` does **not** start the JVM (`docs/architecture.md` §10);
the JVM spins up lazily on the first Orekit-touching call (here, the first
propagation).

In [ ]:
from pathlib import Path

import numpy as np

import propygator as pgr

pgr.__version__

## 1. Build an initial state

A `State` is a Cartesian position + velocity at an `Epoch`, in an explicit `Frame`.
Everything is SI (metres, m/s) and the frame must be inertial for propagation
(`EME2000` / `J2000`). Here we build a circular, ISS-like LEO at ~420 km altitude
and 51.6° inclination.

In [ ]:
MU = 3.986004418e14  # Earth GM (WGS84), m^3/s^2
r = 6798137.0  # ~420 km altitude
v = np.sqrt(MU / r)  # circular speed
inc = np.radians(51.6)

initial = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r, 0.0, 0.0]),
    np.array([0.0, v * np.cos(inc), v * np.sin(inc)]),
    pgr.Frame.EME2000,
)
initial

## 2. Propagate

`propagate_numerical` returns a `Trajectory` in `EME2000`, sampled every
`output_step` seconds. With no `force_models` argument it uses the `leo_default`
preset (gravity field + Sun/Moon third body + drag + SRP + tides). `duration` and
`output_step` are in seconds.

In [ ]:
traj = pgr.propagate_numerical(initial, duration=86400, output_step=60)
len(traj), traj.frame

Every `Trajectory` carries a reproducibility record in `metadata` — versions,
integrator, tolerances, the exact list of forces that acted, and so on.

In [ ]:
dict(traj.metadata)

## 3. Inspect the trajectory

A `Trajectory` indexes and iterates as `State` objects, converts between frames in
bulk, interpolates to an arbitrary epoch with `at()`, and exposes osculating
Keplerian elements via `to_keplerian()`.

In [ ]:
first = traj[0]
kep = first.to_keplerian()
print(f"a = {kep.semi_major_axis_m / 1000:.1f} km")
print(f"e = {kep.eccentricity:.5f}")
print(f"i = {np.degrees(kep.inclination_rad):.2f} deg")

# Bulk frame conversion (EME2000 -> Earth-fixed ITRF).
itrf = traj.to_frame(pgr.Frame.ITRF)
itrf.frame

In [ ]:
# Hermite-interpolate to an epoch between samples (30 s after the start).
mid = pgr.Epoch.from_iso("2024-01-01T00:00:30")
traj.at(mid)

## 4. Configure the force model and spacecraft

`ForceModelConfig` ships three presets (`leo_default`, `geo_default`, `keplerian`)
and is fully customizable. `SpacecraftConfig` carries the mass and geometry; a
`box_and_panels` geometry drives both drag and SRP, and the attitude family
(`LofAligned`, `InPlaneTracking`, `LofOffset`, …) sets the orientation. A
`VariableCd` table gives a density-varying drag coefficient (`sphere_default()` is
shipped).

In [ ]:
box_traj = pgr.propagate_numerical(
    initial,
    duration=86400,
    output_step=60,
    force_models=pgr.ForceModelConfig.leo_default(),
    spacecraft=pgr.SpacecraftConfig(
        mass_kg=420,
        geometry=pgr.SpacecraftGeometry.box_and_panels(
            x_length_m=2.0,
            y_length_m=1.0,
            z_length_m=1.0,
            solar_array_area_m2=10.0,
            drag_coefficient=pgr.VariableCd.sphere_default(),
        ),
    ),
    attitude=pgr.InPlaneTracking(),
)
len(box_traj)

### Per-face drag for a convex box (`BoxFaceCd`)

For a **convex box with no solar arrays**, `BoxFaceCd` resolves the drag coefficient
*per face* — how each of the six faces meets the oncoming flow — instead of applying one
scalar Cd uniformly. In free-molecular flow a convex body never self-shadows, so total
drag is the exact sum of the six per-face contributions. `BoxFaceCd.default()` loads the
shipped per-face table; because the coefficient is per-unit-area, one universal table
serves any convex box or plate.

It is accepted only by `box_and_panels` with `solar_array_area_m2=0` (a convex bus). The
per-face correction matters most for a **high-area-to-mass flat plate flown edge-on** — a
solar / drag sail — where tangential shear on the grazing faces dominates and a single
scalar Cd (even a recalibrated one) under-predicts along-track drag by ~2×. For a bus
flown face-on, nadir-held, or Sun-pointing the effect is negligible. See
`docs/features.md` §1.1 ("Drag-coefficient modeling") for the full design.

In [ ]:
# A thin, high-area-to-mass plate (a drag sail): 1 m^2 face, ~1 cm thick, 2 kg.
sail_traj = pgr.propagate_numerical(
    initial,
    duration=86400,
    output_step=60,
    spacecraft=pgr.SpacecraftConfig(
        mass_kg=2.0,
        geometry=pgr.SpacecraftGeometry.box_and_panels(
            x_length_m=1.0,
            y_length_m=1.0,
            z_length_m=0.01,
            solar_array_area_m2=0.0,  # convex box, no arrays -> BoxFaceCd is valid
            drag_coefficient=pgr.BoxFaceCd.default(),
        ),
    ),
    attitude=pgr.InPlaneTracking(),  # body-X along velocity; holds the plate edge-on
)

# The per-face table records its own reproducible metadata token.
print("spacecraft:", sail_traj.metadata["spacecraft"])
len(sail_traj)

`BoxFaceCd` models a *convex box*, so misuse is caught the moment you build the geometry
— never mid-propagation. A sphere has no per-face incidence, and a paneled box is
non-convex; both raise `ValueError` at construction:

In [ ]:
# Both of these raise at construction, with an actionable message.
bad = {
    "sphere": lambda: pgr.SpacecraftGeometry.sphere(
        1.0, drag_coefficient=pgr.BoxFaceCd.default()
    ),
    "box with arrays": lambda: pgr.SpacecraftGeometry.box_and_panels(
        x_length_m=1.0,
        y_length_m=1.0,
        z_length_m=0.01,
        solar_array_area_m2=10.0,
        drag_coefficient=pgr.BoxFaceCd.default(),
    ),
}
for label, build in bad.items():
    try:
        build()
    except ValueError as err:
        print(f"{label}: {err}")

## 5. Plot

Each `plot_*` returns its native figure (matplotlib for 2-D, Plotly for the
interactive 3-D view), so you can post-process with the native API. `plot_summary`
is the default stacked view: ground track, altitude, and a speed panel per frame.

In [ ]:
pgr.plot_summary(traj)

In [ ]:
pgr.plot_ground_track(traj)

In [ ]:
# Interactive 3-D (Plotly). Use frame=pgr.Frame.ITRF with show_map_overlay=True
# to drape the coastline on a static Earth.
pgr.plot_3d(traj).show()

## 6. Export everything

`export_all` bundles the common case into an output directory: a summary PNG, an
interactive 3-D HTML, and a CSV (16 default columns plus a metadata header). It
returns a dict mapping each output name to the file written. Pass several
`frames_3d` to get one frame-suffixed 3-D file each, or use `export_csv` /
`plot_*` individually for finer control.

In [ ]:
outputs = pgr.export_all(
    traj,
    output_dir=Path("./run_01"),
    speed_frames=[pgr.Frame.EME2000, pgr.Frame.ITRF],
    frames_3d=[pgr.Frame.EME2000, pgr.Frame.ITRF],
)
outputs

## 7 · Altitude guards & graceful termination

The propagator watches the satellite's distance from Earth and **stops cleanly at
physical boundaries** instead of crashing or running forever. Three things can stop a
run early, and all of them *report* rather than raise an error:

- **impact** (`r` reaches the Earth's radius) and **escape** (`r` passes the Earth–Moon
  gravity-parity radius, ~327,000 km) — always on;
- a **drag-driven re-entry** — caught as the orbit decays;
- optional **user altitude limits** you pass in.

When a guard stops a run you get back a *partial* `Trajectory` (samples up to the
crossing) whose `metadata` gains `terminated`, `termination_reason`, and
`termination_epoch`. A normal run carries none of these keys.

In [ ]:
R_EARTH = 6_378_137.0

# A very low orbit (140 km) with drag on decays within hours.
r = R_EARTH + 140_000.0
speed = np.sqrt(MU / r)
decaying = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r, 0.0, 0.0]),
    np.array([0.0, speed, 0.0]),
    pgr.Frame.EME2000,
)

# `duration` is only an upper bound — the run stops itself at re-entry. The one-time
# warning is the drag-validity guard (drag modeling unreliable when low), not an error.
decay_traj = pgr.propagate_numerical(decaying, duration=129_600, output_step=600)

m = decay_traj.metadata
print("terminated        :", m.get("terminated"))
print("termination_reason:", m.get("termination_reason"))
print("termination_epoch :", m.get("termination_epoch"))
print("samples           :", len(decay_traj), "(partial — up to re-entry)")

`terminated` is `True` and `termination_reason` is `"reentry"`. The partial trajectory is
an ordinary `Trajectory` — every verb (`plot_*`, `export_*`, `to_keplerian`) works on it.
For instance, the altitude plot shows the decay:

In [ ]:
pgr.plot_altitude(decay_traj)

### User altitude limits

Pass `limits=` to add your own terminal bounds. They **nest inside** the system backstops
— they can only *tighten* when a run stops, never loosen it. A *reasonable* limit stops &
reports (`"user_min"` / `"user_max"`); an *unreasonable* one (below the surface, or above
escape) could never bind, so it's rejected the moment you construct it.

In [ ]:
# An eccentric orbit falling from a ~522 km apogee toward a ~272 km perigee.
r_apo, r_per = 6.9e6, 6.65e6
a = 0.5 * (r_apo + r_per)
v_apo = np.sqrt(MU * (2.0 / r_apo - 1.0 / a))
eccentric = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([r_apo, 0.0, 0.0]),
    np.array([0.0, v_apo, 0.0]),
    pgr.Frame.EME2000,
)

# Stop if it drops below 350 km (between apogee and perigee). Drag off for a clean demo.
bounded = pgr.propagate_numerical(
    eccentric,
    duration=3000,
    output_step=30,
    force_models=pgr.ForceModelConfig.keplerian(),
    limits=pgr.AltitudeLimits(min_altitude_km=350.0),
)
print(
    "reason:",
    bounded.metadata["termination_reason"],
    "at",
    bounded.metadata["termination_epoch"],
)

# An unreasonable limit is rejected immediately, before anything propagates:
try:
    pgr.AltitudeLimits(min_altitude_km=-50.0)
except ValueError as err:
    print("rejected at construction:", err)

### Escape

An unbound (hyperbolic) orbit is propagated faithfully, then stopped at the escape radius
so it can't run out to absurd distances.

In [ ]:
hyperbolic = pgr.State(
    pgr.Epoch.from_iso("2024-01-01T00:00:00"),
    np.array([7.0e6, 0.0, 0.0]),
    np.array([0.0, 11_500.0, 0.0]),  # faster than escape speed at 7,000 km
    pgr.Frame.EME2000,
)
print(
    "eccentricity:",
    round(hyperbolic.to_keplerian().eccentricity, 2),
    "(> 1 -> unbound)",
)

escape_traj = pgr.propagate_numerical(hyperbolic, duration=10 * 86400, output_step=3600)
print("reason:", escape_traj.metadata["termination_reason"])

---

That's the full Feature 1.1 loop: state → `propagate_numerical` → `Trajectory` → inspect / plot / export, including the altitude guards that stop and report at re-entry, escape, and user limits. The full contract (every preset, validation rule, and output column) lives in `docs/features.md` §1.1; `03_demo.ipynb` runs four contrasting orbits end to end.